In [ ]:
pip install astropy astroquery numpy matplotlib imageio reproject

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 78.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.2 MB/s  0:00:00
  Created wheel for pims: filename=pims-0.7-py3-none-any.whl size=84644 sha256=e23525ff1ddf2a00c24bdf96f580bfa272d5c9976ac6f2f3703babf85cfc8d89
  Stored in directory: /root/.cache/pip/wheels/9b/9e/a8/a0f41cf12a4a7b98a501ad8590595e2071a946d2fd33b0eb60
Successfully built pims
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [astroquery]


In [2]:
"""SPHEREx cutout pipeline: query SIA, fetch cutouts, align north-up.
"""

import functools
import hashlib
import io
import math
import random
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import requests
import astropy.units as u
from astropy.coordinates import Angle, SkyCoord
from astropy.coordinates.name_resolve import NameResolveError
from astropy.io import fits
from astropy import log as astropy_log
from astropy.io.votable import parse_single_table
from astropy.table import Table, vstack
from astropy.time import Time
from astropy.wcs import WCS
from pyvo.dal.exceptions import DALAccessError
from pyvo.dal.sia2 import SIA2Service

try:
    from erfa import ErfaWarning
    warnings.filterwarnings("ignore", category=ErfaWarning)
except ImportError:
    pass

# WCS(hdr, key="W", ...) logs a SIP INFO message on every call.
astropy_log.setLevel("ERROR")

try:
    from reproject import reproject_interp
except ImportError:  # pip install reproject
    reproject_interp = None

SIA_URL = "https://irsa.ipac.caltech.edu/SIA"
COLLECTIONS = ("spherex_dr1", "spherex_qr3")

# SPHEREx has six detectors. Each one sees the same sky strip in a different
# wavelength window, and an exposure writes one row per detector that covered
# the target (two rows: the same column, two halves of the band split).
BANDS = ("SPHEREx-D1", "SPHEREx-D2", "SPHEREx-D3",
         "SPHEREx-D4", "SPHEREx-D5", "SPHEREx-D6")

# Cutouts are ~4 MB each; set to None to use every exposure (needed for stacking).
MAX_FRAMES_PER_BAND = 36

# Photometry.  The PSF FWHM is 3.1" but one pixel is 6.15", so a point source
# is barely resolved: measured on bright stars in these frames, ~80% of the
# flux falls inside r=2 px (12"), and ~85% inside 3 px.  The aperture is
# therefore r=2 px with a fixed correction; it is a relative measurement and is
# contaminated by anything within ~12".
APERTURE_R_PX = 2.0
APERTURE_CORRECTION = 1.25

# Flag bits that are normal, not artefacts. Everything else marks a bad pixel.
BENIGN_FLAGS = {"SOURCE", "FULLSAMPLE"}

In [3]:
# --------------------------------------------------------------------------
# input validation
# --------------------------------------------------------------------------

def parse_ra(text: str) -> float:
    text = text.strip()
    if not text:
        raise ValueError("RA is empty")

    try:
        value = float(text)                      # "133.79513", "nan", "inf"
    except ValueError:
        try:
            value = Angle(text).to_value(u.deg)  # "08h55m10.831s", "-07d14m42.53s"
        except (ValueError, u.UnitsError):
            # colon/space separated: by convention RA is hours
            try:
                value = Angle(text, unit=u.hourangle).to_value(u.deg)
            except (ValueError, u.UnitsError) as exc:
                raise ValueError(f"cannot parse RA {text!r}") from exc

    if not math.isfinite(value):
        raise ValueError(f"RA is not finite: {text!r}")
    value %= 360.0
    return 0.0 if value == 360.0 else value


def parse_dec(text: str) -> float:
    text = text.strip()
    if not text:
        raise ValueError("Dec is empty")

    try:
        value = float(text)
    except ValueError:
        try:
            value = Angle(text).to_value(u.deg)
        except (ValueError, u.UnitsError):
            try:
                value = Angle(text, unit=u.deg).to_value(u.deg)
            except (ValueError, u.UnitsError) as exc:
                raise ValueError(f"cannot parse Dec {text!r}") from exc

    if not math.isfinite(value):
        raise ValueError(f"Dec is not finite: {text!r}")
    if not -90.0 <= value <= 90.0:
        raise ValueError(f"Dec out of range: {value}")
    return value


def resolve_name(name: str) -> SkyCoord:
    name = name.strip()
    if not name:
        raise ValueError("name is empty")
    try:
        coord = SkyCoord.from_name(name)   # cache=True by default
    except NameResolveError as exc:
        raise ValueError(
            f"name not found: {name!r}. Try the canonical designation "
            f"(e.g. 'WISE J085510.83-071442.5') or enter RA/Dec directly."
        ) from exc
    if not (np.isfinite(coord.ra.deg) and np.isfinite(coord.dec.deg)):
        raise ValueError(f"resolved non-finite coordinate for {name!r}")
    return coord


def make_target(ra_text: str, dec_text: str) -> SkyCoord:
    ra, dec = parse_ra(ra_text), parse_dec(dec_text)
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
    if not (np.isfinite(coord.ra.deg) and np.isfinite(coord.dec.deg)):
        raise ValueError("non-finite coordinate")
    return coord


def epoch_position(coord, pm_ra_cosdec, pm_dec, t):
    """Move a J2000 position to time `t` using proper motion (mas/yr)."""
    if pm_ra_cosdec is None or pm_dec is None:
        return coord
    if not hasattr(pm_ra_cosdec, "unit"):
        pm_ra_cosdec = pm_ra_cosdec * u.mas / u.yr
    if not hasattr(pm_dec, "unit"):
        pm_dec = pm_dec * u.mas / u.yr
    c = SkyCoord(ra=coord.ra, dec=coord.dec,
                 pm_ra_cosdec=pm_ra_cosdec, pm_dec=pm_dec,
                 obstime=Time("J2000"), frame="icrs")
    return c.apply_space_motion(new_obstime=t)


# --------------------------------------------------------------------------
# SIA query
# --------------------------------------------------------------------------

def _search_collection(target, radius, collection, attempts=3, timeout=(10, 60)):
    """One SIA2 query with retries. Returns None if every attempt failed."""
    last = None
    for i in range(1, attempts + 1):
        t0 = time.time()
        try:
            sess = requests.Session()
            sess.request = functools.partial(sess.request, timeout=timeout)
            service = SIA2Service(baseurl=SIA_URL, session=sess)
            table = service.search(pos=(target, radius),
                                   collection=[collection]).to_table()
            if len(table) == 0 and i < attempts:
                print(f"  {collection}: empty result, retrying")
                time.sleep(4.0 * i)
                continue
            print(f"  {collection}: {len(table)} rows in {time.time()-t0:.1f}s")
            return table
        except (DALAccessError, requests.RequestException, OSError) as exc:
            last = exc
            print(f"  {collection} [{i}/{attempts}] {type(exc).__name__} "
                  f"after {time.time()-t0:.0f}s: {str(exc)[:70]}")
            if i < attempts:
                time.sleep(min(30.0, 4.0 * i))
    print(f"  {collection}: giving up ({type(last).__name__})")
    return None


def query_sia(target, radius, collections=COLLECTIONS):
    """Query each collection separately: smaller responses, clearer failures."""
    tables = []
    for collection in collections:
        table = _search_collection(target, radius, collection)
        if table is None:
            print(f"  {collection}: falling back to a direct GET")
            table = query_sia_direct(target.ra.deg, target.dec.deg,
                                     radius.to_value(u.deg),
                                     collections=(collection,))
        if table is not None and len(table):
            tables.append(table)

    if not tables:
        raise RuntimeError("SIA unavailable")
    return vstack(tables) if len(tables) > 1 else tables[0]


def query_sia_direct(ra, dec, radius_deg, collections=COLLECTIONS,
                     attempts=3, timeout=(10, 60)):
    """Fallback: plain GET, bypassing pyvo's capabilities call."""
    params = [("COLLECTION", c) for c in collections]
    params += [
        ("POS", f"CIRCLE {ra:.6f} {dec:.6f} {radius_deg:.6f}"),
        ("RESPONSEFORMAT", "votable"),
    ]
    for i in range(1, attempts + 1):
        t0 = time.time()
        try:
            r = requests.get(SIA_URL, params=params, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as exc:
            print(f"[{i}/{attempts}] {type(exc).__name__} after {time.time()-t0:.0f}s")
            time.sleep(5 * i)
            continue

        print(f"[{i}/{attempts}] HTTP {r.status_code} "
              f"{len(r.content)} B in {time.time()-t0:.0f}s")

        if r.status_code == 200:
            # IRSA writes FIELD ID="col_15" alongside name="access_url";
            # astropy's unified reader keys off ID, so re-apply the names.
            vot = parse_single_table(io.BytesIO(r.content))
            table = vot.to_table()
            table.rename_columns(table.colnames, [f.name for f in vot.fields])
            return table
        if r.status_code in (502, 503, 504):
            time.sleep(5 * i)
            continue
        r.raise_for_status()          # 400 = your parameters are wrong

    raise RuntimeError("SIA unavailable")


def get_exposures(target, radius, cache_path):
    if cache_path.exists():
        table = Table.read(cache_path)
        print(f"loaded {len(table)} rows from {cache_path}")
        return table

    try:
        table = query_sia(target, radius)
    except RuntimeError:
        print("pyvo path failed; trying a direct request")
        table = query_sia_direct(target.ra.deg, target.dec.deg, radius.to_value(u.deg))

    table.write(cache_path, format="ecsv", overwrite=True)
    print(f"wrote {len(table)} rows to {cache_path}")
    return table


# --------------------------------------------------------------------------
# cutouts
# --------------------------------------------------------------------------

class NotCovered(Exception):
    """Target is not on this exposure (HTTP 422 from IBE)."""


def fetch_cutout(access_url, ra_deg, dec_deg, size_deg, attempts=3, cache_dir=None):
    url = f"{access_url}?center={ra_deg:.7f},{dec_deg:.7f}&size={size_deg}"

    path = None
    if cache_dir is not None:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        path = Path(cache_dir) / (hashlib.md5(url.encode()).hexdigest() + ".fits")
        if path.exists():
            return fits.open(path)

    for i in range(1, attempts + 1):
        try:
            r = requests.get(url, timeout=120)
        except requests.RequestException:
            if i == attempts:
                raise
            time.sleep(2 ** i)
            continue

        if r.status_code == 200:
            if not r.content.startswith(b"SIMPLE"):
                raise OSError(f"not a FITS file from {access_url}")
            if path is not None:
                path.write_bytes(r.content)
            return fits.open(io.BytesIO(r.content))

        if r.status_code == 422:
            raise NotCovered(f"position off footprint: {access_url}")

        if r.status_code in (502, 503, 504):
            if i == attempts:
                r.raise_for_status()
            time.sleep(5 * i + random.random() * 3)
            continue

        r.raise_for_status()


def flag_names(hdr):
    return {int(hdr[f"MSKM{i:04d}"]): hdr[f"MSKN{i:04d}"].strip().upper()
            for i in range(64) if f"MSKM{i:04d}" in hdr}


def bad_bits(hdr):
    """OR of every flag bit that is not benign (SOURCE / FULLSAMPLE)."""
    total = 0
    for bit, name in flag_names(hdr).items():
        if name not in BENIGN_FLAGS:
            total |= bit
    return total


def band_wavelength_um(row):
    """Band edges from SIA (em_min/em_max are in metres)."""
    return float(row["em_min"]) * 1e6, float(row["em_max"]) * 1e6


def pixel_area_sr(hdr):
    """Pixel solid angle: OMEGA_MEDIAN is in arcsec^2, convert to sr."""
    omega = float(hdr.get("OMEGA_MEDIAN", 37.86))
    return omega * (np.pi / 180.0 / 3600.0) ** 2


def wavelength_at(hdul, wcs_img, ra, dec):
    """Per-pixel wavelength and bandwidth at a sky position.

    The header carries a WAVE-TAB alternate WCS (CRPIX1W, CTYPE1W='WAVE-TAB',
    ...).  It is *not* along x: for D1..D6 the dispersion runs along detector y,
    which is why the 9x9 lookup grid must be read through astropy rather than
    interpolated by hand.
    """
    x, y = wcs_img.world_to_pixel(SkyCoord(ra, dec, unit="deg", frame="icrs"))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # SIP noise from the alternate WCS
        wave_wcs = WCS(hdul["IMAGE"].header, key="W", fobj=hdul)
        lam, dlam = wave_wcs.pixel_to_world(float(x), float(y))
    return float(lam.to_value(u.micron)), float(dlam.to_value(u.micron))


def centroid_offset(image, flags, bad, x, y, r=3.0):
    """Flux-weighted centroid near (x, y).  Returns (dx, dy) in pixels.

    This is a contamination check, not a measurement: at 6.16"/px the PSF is
    ~1.2 px, so anything 1-2 px away is blended into the same aperture.
    """
    h, w = image.shape
    yy, xx = np.mgrid[0:h, 0:w]
    d = np.hypot(xx - x, yy - y)
    good = np.isfinite(image) & ((flags & bad) == 0)
    ann = (d >= 5) & (d <= 9) & good
    if ann.sum() < 10:
        return None
    bkg = float(np.median(image[ann]))
    # np.where, not mask*array: one bad pixel makes 0 * nan = nan and the
    # whole weighted sum becomes nan.
    wgt = np.where((d <= r) & good, np.clip(image - bkg, 0, None), 0.0)
    if wgt.sum() <= 0:
        return None
    return (float((wgt * xx).sum() / wgt.sum() - x),
            float((wgt * yy).sum() / wgt.sum() - y))


def aperture_photometry(image, variance, flags, bad, x, y,
                        r_ap=APERTURE_R_PX, r_in=6.0, r_out=12.0):
    """Aperture sum with a sigma-clipped local background.

    `image` must already have the zodiacal model subtracted: ZODI is 0.2-0.7
    MJy/sr in this field, so any residual gradient across the 13 pixels of the
    aperture is larger than the source being measured.

    Returns (flux, error, background) in MJy/sr summed over pixels, or None.
    The error is the quadrature sum of the per-pixel VARIANCE plus the scatter
    of the local background, so it is honest about sky noise.
    """
    h, w = image.shape
    yy, xx = np.mgrid[0:h, 0:w]
    d = np.hypot(xx - x, yy - y)
    good = (np.isfinite(image) & np.isfinite(variance) & (variance > 0)
            & ((flags & bad) == 0))
    ann = (d >= r_in) & (d <= r_out) & good
    if ann.sum() < 20:
        ann = (d >= 4) & (d <= 15) & good          # small cutouts
    ap = (d <= r_ap) & good
    if ap.sum() == 0 or ann.sum() < 20:
        return None

    vals = image[ann]
    bkg = float(np.median(vals))
    sig_bkg = 1.4826 * float(np.median(np.abs(vals - bkg)))
    flux = float(np.sum(image[ap] - bkg))
    err = float(np.sqrt(np.sum(variance[ap]) + ap.sum() * sig_bkg ** 2))
    return flux, err, bkg, sig_bkg


# --------------------------------------------------------------------------
# alignment
# --------------------------------------------------------------------------

def north_up_wcs(ra, dec, fov_deg, pixel_scale_arcsec=6.1577):
    n = int(round(fov_deg * 3600.0 / pixel_scale_arcsec))
    w = WCS(naxis=2)
    w.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    w.wcs.crpix = [(n + 1) / 2.0, (n + 1) / 2.0]
    w.wcs.crval = [ra, dec]
    s = pixel_scale_arcsec / 3600.0
    w.wcs.cd = np.array([[-s, 0.0], [0.0, s]])   # minus: RA grows to the left
    w.wcs.radesys = "ICRS"
    return w, n


def shifted_wcs(wcs_in, dra, ddec):
    """Shift CRVAL so a star that moved by (dra, ddec) keeps its pixel.

    The displacement is arcsec-scale, so a plain CRVAL offset is accurate to
    well under a pixel even at these declinations.
    """
    w = wcs_in.deepcopy()
    w.wcs.crval = [w.wcs.crval[0] + dra, w.wcs.crval[1] + ddec]
    return w


def centroid_offsets(records, half=5):
    """Centroid of one bright star across frames: a cheap alignment check."""
    stack = np.nanmedian([r["image_r"] for r in records], axis=0)
    h, w = stack.shape
    edge = 12
    core = stack[edge:h - edge, edge:w - edge]
    iy, ix = np.unravel_index(np.nanargmax(core), core.shape)
    iy += edge
    ix += edge

    y0, y1 = max(0, iy - half), min(h, iy + half + 1)
    x0, x1 = max(0, ix - half), min(w, ix + half + 1)
    ys, xs = np.mgrid[y0:y1, x0:x1]

    offs = []
    for r in records:
        img = r["image_r"][y0:y1, x0:x1]
        if not np.all(np.isfinite(img)):
            continue
        wgt = np.clip(img - np.median(img), 0, None)
        if wgt.sum() <= 0:
            continue
        offs.append(((wgt * xs).sum() / wgt.sum() - ix,
                     (wgt * ys).sum() / wgt.sum() - iy))

    if not offs:
        return None, None
    offs = np.array(offs)
    return offs.mean(axis=0), offs.std(axis=0)


def save_contact_sheet(records, path="contact_sheet.png", per_band=4):
    """One row per detector, one column per epoch: the D1..D6 view."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    bands = [b for b in BANDS if any(r["band"] == b for r in records)]
    if not bands:
        return
    cols = max(1, min(per_band, max(sum(r["band"] == b for r in records)
                                     for b in bands)))
    fig, axes = plt.subplots(len(bands), cols,
                             figsize=(3.0 * cols, 2.9 * len(bands)), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")

    for row_i, band in enumerate(bands):
        frames = [r for r in records if r["band"] == band][:cols]
        for col_i, rec in enumerate(frames):
            ax = axes[row_i, col_i]
            img = rec.get("image_r", rec["image"])
            lo, hi = np.nanpercentile(img, [5, 99])
            # uncovered area would otherwise render as white; make it black
            img = np.where(np.isfinite(img), img, lo)
            ax.imshow(img, origin="lower", cmap="gray", vmin=lo, vmax=hi)
            ax.set_title(f"{band[-2:]}  {rec['t'].iso[5:16]}  {rec['lam']:.2f} um",
                         fontsize=8)
            ax.axis("on")
            ax.set_xticks([])
            ax.set_yticks([])

    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)
    print("wrote", path)


def save_spectrum(records, path="spectrum.png"):
    """Flux vs per-frame wavelength: the six detectors on one axis."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for band in BANDS:
        frames = [r for r in records if r["band"] == band and "flux_ujy" in r]
        if not frames:
            continue
        lam = np.array([r["lam"] for r in frames])
        flux = np.array([r["flux_ujy"] for r in frames])
        err = np.array([r["err_ujy"] for r in frames])
        order = np.argsort(lam)
        ax.errorbar(lam[order], flux[order], yerr=err[order], fmt="o", ms=5,
                    capsize=2, label=f"{band[-2:]}")

    ax.axhline(0.0, color="0.7", lw=0.8)
    ax.set_xlabel("wavelength (um)")
    ax.set_ylabel("aperture flux (uJy)")
    ax.set_title("SPHEREx spectrum of the target")
    ax.legend(fontsize=8, ncol=3)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)
    print("wrote", path)

In [ ]:
# --------------------------------------------------------------------------
# main
# --------------------------------------------------------------------------

# Target: a name (resolved through Sesame, which returns ICRS at J2000) or
# explicit J2000 coordinates.  Hand-entered coordinates MUST be J2000:
# designation positions such as "085510.83-071442.5" are quoted at the
# discovery epoch, and for an 8"/yr star that is ~85" from J2000.
input_name = "Polaris"
input_ra = "133.818908"        # J2000, only used if the name does not resolve
input_dec = "-7.247048"

# Proper motion for the current target (mas/yr, ICRS pm_ra_cosdec). WISE 0855.
PM_RA = -8123.7
PM_DEC = 673.2

if input_name:
    try:
        target = resolve_name(input_name)
        print(f"{input_name} -> J2000 {target.ra.deg:.6f} {target.dec.deg:.6f}")
    except ValueError as exc:
        print(exc)
        print("falling back to the J2000 coordinates in input_ra/input_dec")
        target = make_target(input_ra, input_dec)
else:
    target = make_target(input_ra, input_dec)

ra, dec = target.ra.deg, target.dec.deg
print(f"J2000 position: {ra:.6f} {dec:.6f}")

fov = 0.2 * u.deg        # user chooses: what the movie shows

# SIA POS is an INTERSECTS test on each exposure footprint, so a small circle
# already returns every exposure that covers the target. Measured on IRSA:
#   r=0.005 deg -> 258 rows, 2.1 s      r=0.05 deg -> 263 rows, 3.3 s
#   r=0.10  deg -> read timeout
# The radius does NOT set the field of view; the cutout size does.
radius = 0.02 * u.deg

# Query around the epoch-corrected position: these targets move a lot.
t_query = Time.now()
pos_query = epoch_position(target, PM_RA, PM_DEC, t_query)
print(f"epoch {t_query.iso[:10]}: {pos_query.ra.deg:.5f} {pos_query.dec.deg:.5f}")

cache = Path(f"exposures_{ra:.3f}_{dec:.3f}.ecsv".replace("-", "m"))
exposures = get_exposures(pos_query, radius, cache)

print(exposures["obs_id", "energy_bandpassname", "t_min"][:5])
print("rows:", len(exposures))
print("columns:", exposures.colnames)
print("bands:", Counter(exposures["energy_bandpassname"]))
t_min_values = np.asarray(exposures["t_min"], dtype=float)
print("t_min range:", Time(float(np.nanmin(t_min_values)), format="mjd").iso,
      "->", Time(float(np.nanmax(t_min_values)), format="mjd").iso)

# From here on the code treats D1..D6 as six wavelength windows of one field.
t_ref = Time(float(np.nanmedian(t_min_values)), format="mjd")
pos_ref = epoch_position(target, PM_RA, PM_DEC, t_ref)
ra_obs, dec_obs = pos_ref.ra.deg, pos_ref.dec.deg
print(f"\nreference epoch {t_ref.iso[:10]}: {ra_obs:.5f} {dec_obs:.5f}")

cutout_size_deg = round(1.5 * fov.to_value(u.deg), 3)
per_band = MAX_FRAMES_PER_BAND if MAX_FRAMES_PER_BAND else "all"
print(f"cutout {cutout_size_deg} deg, {per_band} frames per band")

bands_present = [b for b in BANDS if np.any(exposures["energy_bandpassname"] == b)]
missing = [b for b in BANDS if b not in bands_present]
if missing:
    print("no exposures in:", ", ".join(missing))

records = []
for band in bands_present:
    rows = exposures[exposures["energy_bandpassname"] == band]
    rows = rows[np.argsort(np.asarray(rows["t_min"], dtype=float))]
    if MAX_FRAMES_PER_BAND is not None:
        rows = rows[:MAX_FRAMES_PER_BAND]
    print(f"\n{band}: {len(rows)} frames")

    for row in rows:
        try:
            hdul = fetch_cutout(row["access_url"], ra_obs, dec_obs, cutout_size_deg,
                                cache_dir="cutout_cache")
        except NotCovered:
            print("  skip (off footprint):", row["obs_id"])
            continue
        except (requests.HTTPError, OSError) as exc:
            print("  skip (HTTP):", row["obs_id"], exc)
            continue

        hdr = hdul["IMAGE"].header
        wcs_in = WCS(hdr)                      # SIP stays on
        lam_lo, lam_hi = band_wavelength_um(row)
        names = flag_names(hdul["FLAGS"].header)

        # Where is the target *in this exposure*?  These stars move up to 8"/yr,
        # so the position has to be re-evaluated per frame, not once.
        t_frame = Time(row["t_min"], format="mjd")
        pos_frame = epoch_position(target, PM_RA, PM_DEC, t_frame)
        x, y = wcs_in.world_to_pixel(pos_frame)
        x, y = float(x), float(y)
        ix, iy = int(round(x)), int(round(y))
        if not (0 <= ix < hdr["NAXIS1"] and 0 <= iy < hdr["NAXIS2"]):
            # IBE clips the cutout at the detector edge and returns a sliver
            # with the target outside it.
            print(f"  skip (target off detector): {row['obs_id']} "
                  f"cutout {hdr['NAXIS1']}x{hdr['NAXIS2']}, pixel ({x:.0f}, {y:.0f})")
            hdul.close()
            continue

        target_flag = int(hdul["FLAGS"].data[iy, ix])
        image_raw = np.array(hdul["IMAGE"].data, dtype=float)
        # ZODI is the zodiacal-light model for this exposure.  It is 0.18-0.68
        # MJy/sr here, i.e. far larger than the target, and it is not flat:
        # without subtracting it, the aperture sum measures the sky gradient.
        zodi = (np.array(hdul["ZODI"].data, dtype=float)
                if "ZODI" in hdul else np.zeros_like(image_raw))
        image = image_raw - zodi
        variance = np.array(hdul["VARIANCE"].data, dtype=float)
        flags = np.array(hdul["FLAGS"].data, dtype=np.int32)
        bad = bad_bits(hdul["FLAGS"].header)

        lam, dlam = wavelength_at(hdul, wcs_in, pos_frame.ra.deg, pos_frame.dec.deg)
        area_sr = pixel_area_sr(hdr)
        phot = aperture_photometry(image, variance, flags, bad, x, y)
        off_px = centroid_offset(image, flags, bad, x, y)
        # arcsec/px, straight from SIA (the WCS helper returns a mixed
        # list/Quantity depending on the file, which is not worth unwrapping).
        pix_scale = float(row["s_pixel_scale"])
        hdul.close()

        rec = dict(
            band=band,
            obs_id=row["obs_id"],
            mjd=float(row["t_min"]),
            t=t_frame,
            lam=lam, dlam=dlam,
            lam_lo=lam_lo, lam_hi=lam_hi,
            image=image, variance=variance, flags=flags,
            wcs=wcs_in,
            bad_bits=bad,
            target_flag=target_flag,
            target_flag_names="+".join(n for b, n in sorted(names.items())
                                       if target_flag & b) or "0",
            dra=pos_frame.ra.deg - ra_obs,
            ddec=pos_frame.dec.deg - dec_obs,
            offset_px=off_px,
            pix_scale_arcsec=pix_scale,
        )
        if phot is not None:
            flux, err, sky, sig_sky = phot
            rec["flux_ujy"] = flux * area_sr * 1e12 * APERTURE_CORRECTION
            rec["err_ujy"] = err * area_sr * 1e12 * APERTURE_CORRECTION
            # Peak-pixel S/N: with a 3.1" PSF in 6.15" pixels this is the
            # honest detection test; an aperture sum over 13 noisy pixels can
            # reach S/N 3 by chance with no source present.
            var_pk = variance[iy, ix] if np.isfinite(variance[iy, ix]) else 0.0
            rec["peak_snr"] = ((image[iy, ix] - sky)
                               / np.sqrt(max(var_pk, 0.0) + sig_sky ** 2))
        records.append(rec)

if not records:
    raise SystemExit("no cutouts retrieved")

records.sort(key=lambda r: (r["band"], r["mjd"]))

print(f"\n{len(records)} frames from {len(bands_present)} bands")
print(f"{'band':4} {'date':11} {'MJD':>10} {'lambda':>8} {'dlam':>6} "
      f"{'flux uJy':>9} {'err':>7} {'S/N':>6} {'peakSNR':>8} {'off arcsec':>10} {'bad %':>6}  flag@target")
for r in records:
    bad_pct = 100.0 * np.mean((r["flags"] & r["bad_bits"]) != 0)
    flux = r.get("flux_ujy", float("nan"))
    err = r.get("err_ujy", float("nan"))
    snr = flux / err if err == err else float("nan")
    off = r["offset_px"]
    off_arcsec = (r["pix_scale_arcsec"] * float(np.hypot(*off))
                  if off is not None else float("nan"))
    print(f"{r['band'][-2:]:4} {r['t'].iso[:10]:11} {r['mjd']:10.3f} "
          f"{r['lam']:8.3f} {r['dlam']:6.3f} "
          f"{flux:9.1f} {err:7.1f} {snr:6.1f} {r.get('peak_snr', float('nan')):8.1f} "
          f"{off_arcsec:10.1f} {bad_pct:6.2f}  "
          f"{r['target_flag']} {r['target_flag_names']}")

print(f"\n{'band':4} {'frames':>6} {'lambda sampled (um)':>28} "
      f"{'median off':>11}  verdict")
for band in bands_present:
    sub = [r for r in records if r["band"] == band]
    if not sub:
        continue
    lams = np.array([r["lam"] for r in sub])
    offs = [r["pix_scale_arcsec"] * float(np.hypot(*r["offset_px"]))
            for r in sub if r["offset_px"] is not None]
    med = float(np.median(offs)) if offs else float("nan")
    # One pixel is 6.16" and the PSF is ~1.2 px, so anything beyond half a
    # pixel is a different object inside the aperture, not a centroid error.
    verdict = ("blended/offset" if med > 0.5 * sub[0]["pix_scale_arcsec"]
               else "on target")
    print(f"{band[-2:]:4} {len(sub):6d} {lams.min():12.3f} -> {lams.max():.3f} "
          f"{med:9.1f}\"  {verdict}")

Polaris -> J2000 37.954561 89.264109
J2000 position: 37.954561 89.264109
epoch 2026-11-15: 33.21145 89.26662
loaded 423 rows from exposures_37.955_89.264.ecsv
      obs_id      energy_bandpassname    t_min    
                                           d      
----------------- ------------------- ------------
2025W36_1A_0157_2          SPHEREx-D1 60919.699000
2025W36_1A_0157_2          SPHEREx-D4 60919.699000
2025W36_1A_0157_3          SPHEREx-D1 60919.700492
2025W36_1A_0157_3          SPHEREx-D4 60919.700492
2025W36_1A_0157_4          SPHEREx-D1 60919.701985
rows: 423
columns: ['s_ra', 's_dec', 'facility_name', 'instrument_name', 'dataproduct_subtype', 'calib_level', 'dataproduct_type', 'energy_bandpassname', 'energy_emband', 'obs_id', 's_resolution', 'em_min', 'em_max', 'em_res_power', 'proposal_title', 'access_url', 'access_format', 'access_estsize', 't_exptime', 's_region', 'obs_collection', 'obs_intent', 'algorithm_name', 'facility_keywords', 'instrument_keywords', 'environment_p

In [ ]:
records[0]["mjd"]

In [ ]:
# --------------------------------------------------------------------------
# align every frame north-up on one common grid
# --------------------------------------------------------------------------
if reproject_interp is None:
    print("\nreproject is not installed: pip install reproject")
    print("frames are still in memory under records[i]['image']")
else:
    w_out, n = north_up_wcs(ra_obs, dec_obs, fov.to_value(u.deg))
    print(f"\noutput grid: {n}x{n} px, 6.1577\"/px, north up")

    for r in records:
        # Register on the moving target: shift CRVAL by the proper-motion
        # displacement so the star lands on the same pixel at every epoch.
        wcs_reg = shifted_wcs(r["wcs"], r["dra"], r["ddec"])
        r["image_r"], r["footprint"] = reproject_interp(
            (r["image"], wcs_reg), w_out, shape_out=(n, n))
        r["var_r"], _ = reproject_interp(
            (r["variance"], wcs_reg), w_out, shape_out=(n, n))
        flags_f, _ = reproject_interp(
            (r["flags"], wcs_reg), w_out, shape_out=(n, n), order="nearest-neighbor")
        # NaN outside the footprint would cast to garbage
        r["flags_r"] = np.nan_to_num(flags_f, nan=0.0).round().astype(np.int32)

    full = [r for r in records if r["footprint"].mean() > 0.99]
    print(f"{len(full)}/{len(records)} frames fully cover the output grid")

    for band in bands_present:
        sub = [r for r in records if r["band"] == band and "image_r" in r]
        if len(sub) < 3:
            continue
        mean_off, scatter = centroid_offsets(sub)
        if mean_off is not None:
            print(f"{band[-2:]}: bright-star centroid offset "
                  f"({mean_off[0]:+.2f}, {mean_off[1]:+.2f}) px, "
                  f"scatter ({scatter[0]:.2f}, {scatter[1]:.2f}) px")

    save_contact_sheet(records)
    save_spectrum(records)

In [ ]:
#DISPLAY VIA THE FREQUENCY DOMAIN
import matplotlib.pyplot as plt

%matplotlib inline
print(sorted(round(r["lam"], 5) for r in records))   # look for a value several frames share
print(len(sorted(round(r["lam"], 5) for r in records)))
target_wavelength, tol = 1.51456, 0.00005                 # change to a value from the list above
target_start = 60918
target_end = 66000

sel = [r for r in records if abs(r["lam"] - target_wavelength) <= tol and r["mjd"] >= target_start and r["mjd"] <= target_end]
print(len(sel), "frames in window")

cube = np.array([r["image_r"] for r in sel])
var = np.array([r["var_r"] for r in sel])
valid = np.isfinite(cube) & np.isfinite(var) & (var > 0)

wgt = np.where(valid, 1.0 / np.where(valid, var, 1.0), 0.0)
with np.errstate(invalid="ignore", divide="ignore"):
    stack = np.sum(np.where(valid, cube, 0.0) * wgt, axis=0) / np.sum(wgt, axis=0)

lo, hi = np.nanpercentile(stack, [1, 99.5])
plt.imshow(stack, origin="lower", cmap="gray", vmin=lo, vmax=hi, interpolation="gaussian")
plt.title(f"{target_wavelength} ± {tol} µm")
plt.colorbar(label = "MJy/sr")
plt.show()


In [ ]:
#DISPLAY VIA THE TIME DOMAIN, A SERIES OF IMAGES, CAN BOT STORED AND DISPLAYED AS A ANIMATION
print(sorted(r["mjd"] for r in records))  # look for a value several frames share
print(len(sorted(r["lam"] for r in records)))
target_wavelength, tol = 3.0, 0.01                  # change to a value from the list above
target_start = 0.0
target_end = 99999.9

In [ ]:
target_wl = input("Enter Target Wavelength (µm), Default = 3.0: ")
target_tol = input("Enter Wavelength tolerance (µm), Default = 0.1: ")

In [ ]:
target_wavelength = float(target_wl) if np.isfinite(float(target_wl)) else target_wavelength
tol = float(target_tol) if np.isfinite(float(target_tol)) else tol
print(f"Target Wavelength: {target_wavelength}")
print(f"Target Tol: {tol}")

In [ ]:
target_start_in = input("Enter Target Start MJD (Default = 0): ")

In [ ]:
target_end_in = input("Enter Target End MJD (Default = 99999): ")

In [ ]:
target_start = float(target_start_in) if float(target_start_in) >= target_start and np.isfinite(float(target_start_in)) else target_start
target_end = float(target_end_in) if float(target_end_in) <= target_end and np.isfinite(float(target_end_in)) else target_end
print(f"Target Start MJD: {target_start}")
print(f"Target End MJD: {target_end}")

In [ ]:
sel = [r for r in records if abs(r["lam"] - target_wavelength) <= tol and r["mjd"] >= target_start and r["mjd"] <= target_end]
print(len(sel), "frames in window")

import math
nbr_images = len(sel)
ncols = 4
# Calculate how many rows
nrows = math.ceil(nbr_images / ncols)
print(nrows)
# Create the dynamic figure grid
fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3))
if nrows == 1 and ncols == 1:
    axs = np.array([axs])
else:
    axs = axs.flatten()

ii = 0
for r in sel:
    cube = np.array(r["image_r"])
    var = np.array(r["var_r"])
    valid = np.isfinite(cube) & np.isfinite(var) & (var > 0)
    wgt = np.where(valid, 1.0 / np.where(valid, var, 1.0), 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        stack = np.where(valid, cube * wgt, 0.0) / wgt

    lo, hi = np.nanpercentile(stack, [1, 99.5])
    im = axs[ii].imshow(stack, origin='lower', cmap='gray', vmin=lo, vmax=hi, interpolation="nearest")
    
    # Label with index and wavelength
    wl = r["lam"]
    mjd = r['mjd']
    axs[ii].set_title(f"MJD: {mjd} ({wl:.5f} µm)", fontsize=10)
    
    fig.colorbar(im, ax=axs[ii], fraction=0.046, pad=0.04)
    ii = ii+1
for jj in range(nbr_images, len(axs)):
    axs[jj].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (IntSlider, IntText, Play, Button, Checkbox,
                        HBox, VBox, Layout, jslink, interactive_output)
from IPython.display import display

all_frames = list(sel)                      # the full set, never modified
assert len(all_frames) > 0, "no frames selected"
active = list(all_frames)                   # the frames currently ticked

def frame_image(r):
    img = np.array(r["image_r"], dtype=float)
    lo, hi = np.nanpercentile(img, [1, 99.5])
    return img, lo, hi

def view(i, v):                             # v is a hidden counter that forces a redraw
    if not active:
        print("No frames selected. Tick at least one.")
        return
    i = min(i, len(active) - 1)
    r = active[i]
    img, lo, hi = frame_image(r)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(img, origin="lower", cmap="gray", vmin=lo, vmax=hi, interpolation="nearest")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="MJy/sr")
    ax.set_title(f"{i+1}/{len(active)}   MJD {r['mjd']:.5f}   ({r['lam']:.4f} µm)", fontsize=10)
    plt.show()
    plt.close(fig)

# ---- the frame list: one checkbox per frame ----
boxes = [Checkbox(value=True, indent=False, layout=Layout(width="auto"),
                  description=f"#{k+1}  MJD {r['mjd']:.5f}  {r['lam']:.4f} µm")
         for k, r in enumerate(all_frames)]
frame_list = VBox(boxes, layout=Layout(height="320px", overflow="auto", border="1px solid #ccc"))
btn_all = Button(description="Select all")
btn_none = Button(description="Select none")

# ---- the viewer controls ----
slider = IntSlider(min=0, max=len(active) - 1, step=1, value=0, description="Frame", continuous_update=False)
play = Play(min=0, max=len(active) - 1, step=1, value=0, interval=700)
jslink((play, "value"), (slider, "value"))
prev_btn = Button(description="Previous", icon="step-backward")
next_btn = Button(description="Next", icon="step-forward")
version = IntText(value=0)                  # bumped whenever the selection changes

def step(delta):
    if active:
        slider.value = (slider.value + delta) % len(active)
prev_btn.on_click(lambda b: step(-1))
next_btn.on_click(lambda b: step(+1))

if "playing" in play.trait_names():
    def on_playing(change):
        prev_btn.disabled = next_btn.disabled = bool(change["new"])
    play.observe(on_playing, names="playing")

# ---- apply the ticks ----
def refresh(change=None):
    active[:] = [r for r, b in zip(all_frames, boxes) if b.value]
    top = max(len(active) - 1, 0)
    slider.max = top
    play.max = top
    version.value += 1                      # forces a redraw even if the slider did not move

for b in boxes:
    b.observe(refresh, names="value")
btn_all.on_click(lambda _: [setattr(b, "value", True) for b in boxes])
btn_none.on_click(lambda _: [setattr(b, "value", False) for b in boxes])

out = interactive_output(view, {"i": slider, "v": version})
display(HBox([VBox([HBox([btn_all, btn_none]), frame_list]),
              VBox([HBox([play, prev_btn, next_btn, slider]), out])]))